In [ ]:
#============================================================
# Celda 1 — Setup y carga de datos
#============================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

OUTPUT = Path("output")
FIGS   = OUTPUT / "05_descriptivo" / "figs"
FIGS.mkdir(parents=True, exist_ok=True)

# Cargar datasets
master = pd.read_parquet(OUTPUT / "merged/master_provincia_anio.parquet")
evr    = pd.read_parquet(OUTPUT / "evr/03-gold/gold_evr_provincia_anio.parquet")
eco    = pd.read_parquet(OUTPUT / "economico/03-gold/gold_economico_provincia_anio.parquet")
con    = pd.read_parquet(OUTPUT / "conectividad/03-gold/gold_conectividad_provincia_anio.parquet")

# Estilo global
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

print(f"✅ Master:       {master.shape}")
print(f"✅ EVR gold:     {evr.shape}")
print(f"✅ Económico:    {eco.shape}")
print(f"✅ Conectividad: {con.shape}")

In [ ]:
#============================================================
# Celda 2 — Resumen estadístico del master
#============================================================
cols_interes = [
    "entradas", "salidas", "saldo_neto", "tasa_atraccion",
    "compraventas_vivienda", "poblacion_provincia",
    "viajeros", "cob_100mbps_pct"
]

resumen = master[cols_interes].describe().round(2)
print("📊 Estadísticas descriptivas — Master (2008–2021, 52 provincias)\n")
print(resumen.to_string())

In [ ]:
#============================================================
# Celda 3 — Evolución nacional del saldo neto agregado
#============================================================
saldo_nacional = (
    evr.groupby("anyo")[["entradas", "salidas", "saldo_neto"]]
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    saldo_nacional["anyo"],
    saldo_nacional["saldo_neto"],
    color=["#c0392b" if v < 0 else "#27ae60" for v in saldo_nacional["saldo_neto"]],
    width=0.7, edgecolor="white"
)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Saldo migratorio interprovincial neto agregado — España (2008–2021)")
ax.set_xlabel("Año")
ax.set_ylabel("Saldo neto (personas)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xticks(saldo_nacional["anyo"])

# Anotar periodos clave
ax.axvspan(2008, 2013.5, alpha=0.05, color="red", label="Crisis financiera")
ax.axvspan(2019.5, 2021.5, alpha=0.05, color="blue", label="COVID-19")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / "01_saldo_nacional_anio.png")
plt.show()
print("✅ Guardado: 01_saldo_nacional_anio.png")

In [ ]:
#============================================================
# Celda 4 — Top 10 provincias ganadoras y perdedoras
#============================================================
media_provincia = (
    evr.groupby("provincia")["saldo_neto"]
    .mean()
    .sort_values()
    .reset_index()
)

top_ganadoras  = media_provincia.tail(10).iloc[::-1]
top_perdedoras = media_provincia.head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Ganadoras
axes[0].barh(top_ganadoras["provincia"], top_ganadoras["saldo_neto"],
             color="#27ae60", edgecolor="white")
axes[0].set_title("Top 10 provincias GANADORAS\n(saldo neto medio 2008–2021)")
axes[0].set_xlabel("Saldo neto medio (personas/año)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Perdedoras
axes[1].barh(top_perdedoras["provincia"], top_perdedoras["saldo_neto"],
             color="#c0392b", edgecolor="white")
axes[1].set_title("Top 10 provincias PERDEDORAS\n(saldo neto medio 2008–2021)")
axes[1].set_xlabel("Saldo neto medio (personas/año)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig(FIGS / "02_top10_ganadoras_perdedoras.png")
plt.show()
print("✅ Guardado: 02_top10_ganadoras_perdedoras.png")

In [ ]:
#============================================================
# Celda 5 — Serie temporal top 5 ganadoras vs perdedoras
#============================================================
top5_ganadoras  = media_provincia.tail(5)["provincia"].tolist()
top5_perdedoras = media_provincia.head(5)["provincia"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

colores_g = ["#1a7a4a","#27ae60","#52be80","#82e0aa","#abebc6"]
colores_p = ["#922b21","#c0392b","#e74c3c","#f1948a","#fadbd8"]

for i, prov in enumerate(top5_ganadoras):
    d = evr[evr["provincia"] == prov].sort_values("anyo")
    axes[0].plot(d["anyo"], d["saldo_neto"], marker="o", markersize=4,
                 label=prov, color=colores_g[i], linewidth=1.8)

for i, prov in enumerate(top5_perdedoras):
    d = evr[evr["provincia"] == prov].sort_values("anyo")
    axes[1].plot(d["anyo"], d["saldo_neto"], marker="o", markersize=4,
                 label=prov, color=colores_p[i], linewidth=1.8)

for ax, titulo in zip(axes, ["Top 5 GANADORAS", "Top 5 PERDEDORAS"]):
    ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
    ax.set_title(f"Evolución saldo neto — {titulo} (2008–2021)")
    ax.set_xlabel("Año")
    ax.set_ylabel("Saldo neto (personas)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.legend(fontsize=8)
    ax.set_xticks(range(2008, 2022, 2))

plt.tight_layout()
plt.savefig(FIGS / "03_serie_temporal_top5.png")
plt.show()
print("✅ Guardado: 03_serie_temporal_top5.png")

In [ ]:
#============================================================
# Celda 6 — Distribución saldo neto por año (boxplot)
#============================================================
fig, ax = plt.subplots(figsize=(13, 5))

años = sorted(evr["anyo"].unique())
datos_box = [evr[evr["anyo"] == a]["saldo_neto"].values for a in años]

bp = ax.boxplot(datos_box, labels=años, patch_artist=True,
                medianprops=dict(color="black", linewidth=1.5),
                flierprops=dict(marker=".", markersize=3, alpha=0.5))

for patch in bp["boxes"]:
    patch.set_facecolor("#aed6f1")
    patch.set_alpha(0.7)

ax.axhline(0, color="red", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_title("Distribución del saldo neto interprovincial por año (52 provincias)")
ax.set_xlabel("Año")
ax.set_ylabel("Saldo neto (personas)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig(FIGS / "04_boxplot_saldo_anio.png")
plt.show()
print("✅ Guardado: 04_boxplot_saldo_anio.png")

In [ ]:
#============================================================
# Celda 7 — Evolución compraventas vivienda top 10 provincias
#============================================================
top10_compra = (
    eco.groupby("provincia")["compraventas_vivienda"]
    .mean()
    .nlargest(10)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(12, 5))

for prov in top10_compra:
    d = eco[eco["provincia"] == prov].sort_values("anyo")
    ax.plot(d["anyo"], d["compraventas_vivienda"], marker="o",
            markersize=3, linewidth=1.5, label=prov)

ax.set_title("Evolución compraventas de vivienda — Top 10 provincias (2007–2024)")
ax.set_xlabel("Año")
ax.set_ylabel("Compraventas (nº operaciones)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(fontsize=8, ncol=2)
ax.set_xticks(range(2007, 2025, 2))

plt.tight_layout()
plt.savefig(FIGS / "05_compraventas_top10.png")
plt.show()
print("✅ Guardado: 05_compraventas_top10.png")

In [ ]:
#============================================================
# Celda 8 — Evolución cobertura 100Mbps por CCAA
#============================================================
cob_ccaa = (
    con.groupby(["ccaa", "anyo"])["cob_100mbps_pct"]
    .mean()
    .reset_index()
)

# Top 5 y bottom 5 CCAA por cobertura media
ranking_ccaa = (
    cob_ccaa.groupby("ccaa")["cob_100mbps_pct"]
    .mean()
    .sort_values()
)
bottom5 = ranking_ccaa.head(5).index.tolist()
top5    = ranking_ccaa.tail(5).index.tolist()
seleccion = bottom5 + top5

fig, ax = plt.subplots(figsize=(12, 5))

for ccaa in seleccion:
    d = cob_ccaa[cob_ccaa["ccaa"] == ccaa].sort_values("anyo")
    estilo = "-" if ccaa in top5 else "--"
    ax.plot(d["anyo"], d["cob_100mbps_pct"], marker="o", markersize=4,
            linewidth=1.8, linestyle=estilo, label=ccaa)

ax.set_title("Evolución cobertura 100Mbps — Top 5 vs Bottom 5 CCAA (2013–2024)")
ax.set_xlabel("Año")
ax.set_ylabel("Cobertura (%)")
ax.set_ylim(0, 105)
ax.axhline(50, color="gray", linewidth=0.7, linestyle=":", alpha=0.6)
ax.legend(fontsize=8, ncol=2)
ax.set_xticks(range(2013, 2025, 1))

plt.tight_layout()
plt.savefig(FIGS / "06_cobertura_ccaa_top_bottom.png")
plt.show()
print("✅ Guardado: 06_cobertura_ccaa_top_bottom.png")

In [ ]:
#============================================================
# Celda 9 — Impacto COVID: comparativa 2019 / 2020 / 2021
#============================================================
covid = (
    evr[evr["anyo"].isin([2019, 2020, 2021])]
    .pivot_table(index="provincia", columns="anyo", values="saldo_neto")
    .reset_index()
)
covid.columns.name = None
covid.columns = ["provincia", "saldo_2019", "saldo_2020", "saldo_2021"]
covid["cambio_covid"] = covid["saldo_2020"] - covid["saldo_2019"]
covid = covid.sort_values("cambio_covid")

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(
    covid["provincia"], covid["cambio_covid"],
    color=["#c0392b" if v < 0 else "#27ae60" for v in covid["cambio_covid"]],
    edgecolor="white", height=0.7
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Cambio en saldo neto por provincia: 2020 vs 2019\n(impacto COVID-19)")
ax.set_xlabel("Δ saldo neto (personas)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig(FIGS / "07_impacto_covid_provincia.png")
plt.show()
print("✅ Guardado: 07_impacto_covid_provincia.png")

In [ ]:
#============================================================
# Celda 10 — Resumen y exportar tablas de referencia
#============================================================
TABLES = OUTPUT / "05_descriptivo" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

# Tabla 1: ranking medio de provincias
ranking_medio = (
    evr.groupby("provincia")
    .agg(
        saldo_medio=("saldo_neto", "mean"),
        entradas_media=("entradas", "mean"),
        salidas_media=("salidas", "mean"),
        tasa_atraccion_media=("tasa_atraccion", "mean"),
    )
    .round(1)
    .sort_values("saldo_medio", ascending=False)
    .reset_index()
)
ranking_medio.to_csv(TABLES / "ranking_provincias_saldo_medio.csv", index=False)

# Tabla 2: impacto COVID
covid.to_csv(TABLES / "impacto_covid_provincia.csv", index=False)

# Tabla 3: estadísticas descriptivas master
master.describe().round(2).to_csv(TABLES / "estadisticas_master.csv")

print("✅ Tablas exportadas:")
print(f"   - ranking_provincias_saldo_medio.csv ({len(ranking_medio)} filas)")
print(f"   - impacto_covid_provincia.csv ({len(covid)} filas)")
print(f"   - estadisticas_master.csv")
print(f"\n📁 Figuras en: {FIGS}")
print(f"📁 Tablas en:  {TABLES}")
print(f"\n🏁 05_analisis_descriptivo completado")
print(f"   Figuras generadas: 7")
print(f"   Tablas exportadas: 3")